In [1]:
import pandas as pd
from pathlib import Path

RAW = Path("../data/raw/transfermarkt")

games = pd.read_csv(RAW / "games.csv.gz")
print(games.shape)
print(games.columns.tolist())

(88958, 23)
['game_id', 'competition_id', 'season', 'round', 'date', 'home_club_id', 'away_club_id', 'home_club_goals', 'away_club_goals', 'home_club_position', 'away_club_position', 'home_club_manager_name', 'away_club_manager_name', 'stadium', 'attendance', 'referee', 'url', 'home_club_formation', 'away_club_formation', 'home_club_name', 'away_club_name', 'aggregate', 'competition_type']


In [2]:
pl = games[(games["competition_id"] == "GB1") & games["season"].between(2014, 2025)]

print("Matches per season:")
print(pl.groupby("season").size().to_string())

print("\nMissing manager names:")
print(pl[["home_club_manager_name", "away_club_manager_name"]].isna().sum().to_string())

pl[["date", "home_club_name", "away_club_name", "home_club_goals",
    "away_club_goals", "home_club_manager_name", "away_club_manager_name"]].head()

Matches per season:
season
2014    380
2015    380
2016    380
2017    380
2018    380
2019    380
2020    380
2021    380
2022    380
2023    380
2024    380
2025    380

Missing manager names:
home_club_manager_name    0
away_club_manager_name    0


,date,home_club_name,away_club_name,home_club_goals,away_club_goals,home_club_manager_name,away_club_manager_name
11759,2014-08-16,Arsenal FC,Crystal Palace,2,1,Arsène Wenger,Keith Millen
11760,2014-08-18,Burnley FC,Chelsea FC,1,3,Sean Dyche,José Mourinho
11761,2014-08-16,Leicester City,Everton FC,2,2,Nigel Pearson,Roberto Martínez
11762,2014-08-17,Liverpool FC,Southampton FC,2,1,Brendan Rodgers,Ronald Koeman
11763,2014-08-16,Manchester United,Swansea City,1,2,Louis van Gaal,Garry Monk


In [3]:
transfers = pd.read_csv(RAW / "transfers.csv.gz")
print(transfers.shape)
print(transfers.dtypes)
transfers.head(10)

(175165, 10)
player_id                int64
transfer_date              str
transfer_season            str
from_club_id             int64
to_club_id               int64
from_club_name             str
to_club_name               str
transfer_fee           float64
market_value_in_eur    float64
player_name                str
dtype: object


,player_id,transfer_date,transfer_season,from_club_id,to_club_id,from_club_name,to_club_name,transfer_fee,market_value_in_eur,player_name
0,467994,2030-06-30,25/26,5621,749,Reggiana,FC Empoli,0.0,700000.0,Luca Belardinelli
1,645842,2028-02-02,27/28,6505,19684,Gimcheon Sangmu,Jeju SK,0.0,150000.0,Chan-gi An
2,677470,2028-02-02,27/28,6505,3535,Gimcheon Sangmu,Ulsan HD,0.0,400000.0,Yool Heo
3,709155,2028-02-02,27/28,6505,19684,Gimcheon Sangmu,Jeju SK,0.0,325000.0,Jae-hyeok Oh
4,728133,2028-02-02,27/28,6505,35759,Gimcheon Sangmu,Bucheon FC,0.0,175000.0,Sang-hyeok Lee
5,730364,2028-02-02,27/28,6505,38898,Gimcheon Sangmu,FC Anyang,0.0,150000.0,Gyu-hyeon Choe
6,746780,2028-02-02,27/28,6505,2996,Gimcheon Sangmu,Incheon Utd.,0.0,125000.0,Young-hun Kang
7,755355,2028-02-02,27/28,6505,30925,Gimcheon Sangmu,Gwangju FC,0.0,350000.0,Jin-ho Kim
8,862094,2028-02-02,27/28,6505,21459,Gimcheon Sangmu,Gangwon FC,0.0,400000.0,Jun-seok Song
9,929007,2028-02-02,27/28,6505,19684,Gimcheon Sangmu,Jeju SK,0.0,350000.0,Jun-ha Kim


In [4]:
print("Transfermarkt club names (Premier League, 2014-2025):")
print(sorted(set(pl["home_club_name"])))

pl_club_ids = set(pl["home_club_id"])
incoming = transfers[transfers["to_club_id"].isin(pl_club_ids)]

print(f"\nTransfers INTO clubs that were ever in the PL: {len(incoming):,}")
print("\nBy season:")
print(incoming["transfer_season"].value_counts().sort_index().to_string())

fee = incoming["transfer_fee"]
print(f"\nFee missing (NaN): {fee.isna().sum():,}")
print(f"Fee = 0:           {(fee == 0).sum():,}")
print(f"Fee > 0:           {(fee > 0).sum():,}")

Transfermarkt club names (Premier League, 2014-2025):
['AFC Bournemouth', 'Arsenal FC', 'Aston Villa', 'Brentford FC', 'Brighton & Hove Albion', 'Burnley FC', 'Cardiff City', 'Chelsea FC', 'Crystal Palace', 'Everton FC', 'Fulham FC', 'Huddersfield Town', 'Hull City', 'Ipswich Town', 'Leeds United', 'Leicester City', 'Liverpool FC', 'Luton Town', 'Manchester City', 'Manchester United', 'Middlesbrough FC', 'Newcastle United', 'Norwich City', 'Nottingham Forest', 'Queens Park Rangers', 'Sheffield United', 'Southampton FC', 'Stoke City', 'Sunderland AFC', 'Swansea City', 'Tottenham Hotspur', 'Watford FC', 'West Bromwich Albion', 'West Ham United', 'Wolverhampton Wanderers']

Transfers INTO clubs that were ever in the PL: 5,652

By season:
transfer_season
02/03      1
03/04      4
04/05      5
05/06     12
06/07     23
07/08     31
08/09     42
09/10     50
10/11     97
11/12    127
12/13    127
13/14    184
14/15    220
15/16    259
16/17    287
17/18    352
18/19    363
19/20    378
20/21

In [5]:
window = incoming[incoming["transfer_season"].between("14/15", "25/26")]
paid = window[window["transfer_fee"] > 0]

summary = paid.groupby("transfer_season")["transfer_fee"].agg(
    paid_transfers="count",
    total_eur_m=lambda s: round(s.sum() / 1_000_000),
)
print("Paid transfers INTO PL clubs, by season:")
print(summary.to_string())

famous = ["Di Mar", "Alexis S", "Diego Costa", "De Bruyne", "Pogba", "van Dijk", "Declan Rice"]
spot = incoming[incoming["player_name"].str.contains("|".join(famous), case=False, na=False)]
print("\nSpot-check of famous signings:")
print(spot[["player_name", "transfer_date", "from_club_name", "to_club_name", "transfer_fee"]]
      .sort_values("transfer_date").to_string())

Paid transfers INTO PL clubs, by season:
                 paid_transfers  total_eur_m
transfer_season                             
14/15                        70          803
15/16                       100         1069
16/17                       106         1336
17/18                       118         1964
18/19                       106         1518
19/20                       105         1756
20/21                        94         1504
21/22                       109         1665
22/23                       178         3240
23/24                       162         3084
24/25                       181         3008
25/26                       180         4145

Spot-check of famous signings:
            player_name transfer_date from_club_name to_club_name  transfer_fee
162895  Kevin De Bruyne    2012-01-31           Genk      Chelsea     8000000.0
162424  Kevin De Bruyne    2012-06-30           Genk      Chelsea           0.0
158339  Kevin De Bruyne    2013-06-30  Werder Bremen     

In [6]:
# 1. Which "from" clubs appear most for zero/unknown-fee moves into PL clubs?
zero_or_nan = window[(window["transfer_fee"] == 0) | window["transfer_fee"].isna()]
print("Most common FROM clubs for zero/unknown-fee moves:")
print(zero_or_nan["from_club_name"].value_counts().head(25).to_string())

# 2. How many moves are a player returning to a club he previously left?
t = transfers.copy()
t["transfer_date"] = pd.to_datetime(t["transfer_date"])
t = t.sort_values(["player_id", "transfer_date"])
t["prev_from_club_id"] = t.groupby("player_id")["from_club_id"].shift()
t["looks_like_return"] = (t["to_club_id"] == t["prev_from_club_id"]) & (t["transfer_fee"].fillna(0) == 0)

w = t[t["to_club_id"].isin(pl_club_ids) & t["transfer_season"].between("14/15", "25/26")]
print("\nLooks like a loan return?")
print(w["looks_like_return"].value_counts().to_string())

Most common FROM clubs for zero/unknown-fee moves:
from_club_name
Chelsea          46
Aston Villa      38
Middlesbrough    35
Bournemouth      35
Everton          34
Newcastle        34
Southampton      34
Nott'm Forest    34
Man Utd          33
Without Club     33
Stoke City       32
Swansea          29
Arsenal          29
Sheff Utd        28
Wolves           28
Tottenham        28
Liverpool        27
Burnley          26
West Ham         26
Chelsea U21      26
Brighton         26
Ipswich          26
Leicester        25
Fulham           25
Reading          25

Looks like a loan return?
looks_like_return
False    2980
True     1947
